In [45]:
import pandas as pd
import numpy as np
sales= pd.read_csv("Final_Feature_code.csv")
#sales.dropna(inplace= True)
#sales.isnull().sum()

In [46]:
sales.shape

(100, 1033)

In [47]:
sales.head(100)

,Unnamed: 0,customer_id,order_date,purchase_count,avg_quantity,max_quantity,p25_quantity,p50_quantity,p75_quantity,p90_quantity,...,max_pp_30d_mean,max_pp_30d_min,max_pp_30d_max,max_pp_30d_std,max_pp_30d_var,max_pp_30d_p25,max_pp_30d_p50,max_pp_30d_p75,max_pp_30d_p90,max_pp_30d_p95
0,0,804,2026-04-21,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1079,2026-01-12,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1325,2026-01-18,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1987,2026-02-01,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2593,2026-01-09,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,95210,2026-04-03,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,96,96627,2026-03-05,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,97,96764,2026-01-18,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,98,96876,2026-01-23,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
sales.fillna(sales.median(numeric_only=True), inplace=True)
sales.dropna(inplace= True)
sales.shape

(0, 1033)

In [49]:
#sales.drop(*['order_date', 'last_order_date'],axis=1, inplace= True)
sales.drop('order_date',axis=1, inplace= True)
sales.drop('customer_id',axis=1, inplace= True)
sales.drop('Unnamed: 0',axis=1, inplace= True)

In [50]:
sales.columns

Index(['purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity',
       'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
       'total_spend', 'avg_spend',
       ...
       'max_pp_30d_mean', 'max_pp_30d_min', 'max_pp_30d_max', 'max_pp_30d_std',
       'max_pp_30d_var', 'max_pp_30d_p25', 'max_pp_30d_p50', 'max_pp_30d_p75',
       'max_pp_30d_p90', 'max_pp_30d_p95'],
      dtype='object', length=1030)

In [51]:
sales.shape

(0, 1030)

In [54]:
# ============================================================
# 1. BASIC SETUP
# ============================================================

TARGET = "product_id"

TEST_SIZE = 0.20
RANDOM_STATE = 42

df = sales.copy()

print("=" * 80)
print("DATA VALIDATION")
print("=" * 80)

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# Check target
# ------------------------------------------------------------

if TARGET not in df.columns:
    raise ValueError(
        f"'{TARGET}' does not exist in sales dataframe."
    )

print("\nTarget dtype:")
print(df[TARGET].dtype)

print("\nTarget missing values:")
print(df[TARGET].isna().sum())

print("\nTarget unique values:")
print(df[TARGET].nunique())

print("\nTarget sample:")
print(df[TARGET].head(20))


# ============================================================
# 2. REMOVE ONLY NULL TARGET ROWS
# ============================================================

df = df.dropna(
    subset=[TARGET]
).copy()

print("\nAfter removing NULL target:")
print(df.shape)


# ============================================================
# 3. TARGET DISTRIBUTION
# ============================================================

class_counts = (
    df[TARGET]
    .value_counts()
)

print("\nNumber of product classes:")
print(df[TARGET].nunique())

print("\nSmallest classes:")
print(class_counts.sort_values().head(20))

print("\nLargest classes:")
print(class_counts.sort_values(ascending=False).head(20))


# ============================================================
# 4. CHECK CLASSES WITH ONLY ONE ROW
# ============================================================

single_row_classes = class_counts[
    class_counts < 2
]

print(
    "\nClasses with < 2 observations:",
    len(single_row_classes)
)

print(
    "Rows belonging to those classes:",
    single_row_classes.sum()
)


# ============================================================
# 5. REMOVE ONLY SINGLETON CLASSES
# ============================================================

if len(single_row_classes) > 0:

    valid_classes = class_counts[
        class_counts >= 2
    ].index

    df = df[
        df[TARGET].isin(valid_classes)
    ].copy()


print("\nAfter removing singleton classes:")
print("Rows:", len(df))
print("Classes:", df[TARGET].nunique())


# ============================================================
# IMPORTANT VALIDATION
# ============================================================

if len(df) == 0:

    raise ValueError(
        """
        DATAFRAME HAS ZERO ROWS AFTER PREPROCESSING.

        This means every product_id has fewer than
        2 observations.

        Check:

            sales[TARGET].value_counts()

        If you really have ~500 products but only one
        row per product, you CANNOT train a multiclass
        model to predict product_id from that dataset.
        """
    )


# ============================================================
# 6. CHECK MINIMUM CLASS SIZE
# ============================================================

class_counts = df[TARGET].value_counts()

print("\nMinimum class size:")
print(class_counts.min())

print("\nClass distribution:")
print(class_counts.describe())


# ============================================================
# 7. CREATE X / Y
# ============================================================

X = df.drop(
    columns=[TARGET]
).copy()

y = df[TARGET].copy()


print("\nX shape:")
print(X.shape)

print("\ny shape:")
print(y.shape)


# ============================================================
# 8. DATETIME PROCESSING
# ============================================================

datetime_cols = X.select_dtypes(
    include=[
        "datetime64[ns]",
        "datetime64[ns, UTC]"
    ]
).columns.tolist()

print("\nDatetime columns:")
print(datetime_cols)


for col in datetime_cols:

    X[col + "_year"] = X[col].dt.year

    X[col + "_month"] = X[col].dt.month

    X[col + "_day"] = X[col].dt.day

    X[col + "_dayofweek"] = X[col].dt.dayofweek

    X[col + "_dayofyear"] = X[col].dt.dayofyear

    X[col + "_week"] = (
        X[col]
        .dt.isocalendar()
        .week
        .astype(float)
    )

    X[col + "_hour"] = X[col].dt.hour

    X[col + "_is_weekend"] = (
        X[col].dt.dayofweek >= 5
    ).astype(int)

    X.drop(
        columns=[col],
        inplace=True
    )


# ============================================================
# 9. CATEGORICAL FEATURES
# ============================================================

categorical_cols = X.select_dtypes(
    include=[
        "object",
        "category"
    ]
).columns.tolist()

print("\nCategorical columns:")
print(categorical_cols)


for col in categorical_cols:

    X[col] = X[col].astype("category")


# ============================================================
# 10. INFINITY
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 11. FINAL DATA CHECK
# ============================================================

print("\nFinal X shape:")
print(X.shape)

print("\nFinal y shape:")
print(y.shape)

print("\nFinal classes:")
print(y.nunique())

if len(X) == 0:
    raise ValueError("X has ZERO rows.")

if len(y) == 0:
    raise ValueError("y has ZERO rows.")

if y.nunique() < 2:
    raise ValueError(
        "Multiclass model requires at least 2 classes."
    )


# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 80)
print("TRAIN / TEST SPLIT")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=RANDOM_STATE,

    stratify=y
)


print("\nTRAIN:")
print(X_train.shape)

print("\nTEST:")
print(X_test.shape)

print("\nTrain classes:")
print(y_train.nunique())

print("\nTest classes:")
print(y_test.nunique())


# ============================================================
# 13. CHECK CLASS COVERAGE
# ============================================================

train_classes = set(
    y_train.unique()
)

test_classes = set(
    y_test.unique()
)

missing_train = (
    test_classes - train_classes
)

print(
    "\nClasses in test but missing from train:",
    len(missing_train)
)

if len(missing_train) > 0:

    print(
        missing_train
    )

else:

    print(
        "All test classes exist in training."
    )


# ============================================================
# 14. MULTICLASS LIGHTGBM
# ============================================================

NUM_CLASSES = y.nunique()

print("\n" + "=" * 80)
print("LIGHTGBM")
print("=" * 80)

print(
    "Number of classes:",
    NUM_CLASSES
)


model = LGBMClassifier(

    objective="multiclass",

    num_class=NUM_CLASSES,

    n_estimators=500,

    learning_rate=0.03,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=1.0,

    reg_lambda=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbosity=-1
)


# ============================================================
# 15. TRAIN
# ============================================================

print("\nTraining LightGBM...")

model.fit(

    X_train,

    y_train,

    categorical_feature=categorical_cols
)

print(
    "Training completed."
)


# ============================================================
# 16. PREDICTION
# ============================================================

y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)

print("\nProbability shape:")
print(y_prob.shape)

print(
    "Expected:",
    (
        len(X_test),
        NUM_CLASSES
    )
)


# ============================================================
# 17. PERFORMANCE
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred
)

print("\n" + "=" * 80)
print("MODEL PERFORMANCE")
print("=" * 80)

print(
    "\nAccuracy:",
    round(accuracy, 4)
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy,
        4
    )
)

print("\nClassification report:")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 18. TOP-K ACCURACY
# ============================================================

classes = model.classes_

for k in [3, 5, 10]:

    try:

        score = top_k_accuracy_score(

            y_test,

            y_prob,

            k=k,

            labels=classes

        )

        print(
            f"Top-{k} Accuracy:",
            round(score, 4)
        )

    except Exception as e:

        print(
            f"Top-{k} unavailable:",
            e
        )


# ============================================================
# 19. SHAP SAMPLE
# ============================================================

SHAP_SAMPLE_SIZE = 10000

if len(X_test) > SHAP_SAMPLE_SIZE:

    X_shap = X_test.sample(
        n=SHAP_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

else:

    X_shap = X_test.copy()


print("\n" + "=" * 80)
print("SHAP")
print("=" * 80)

print(
    "SHAP rows:",
    len(X_shap)
)


# ============================================================
# 20. SHAP EXPLAINER
# ============================================================

explainer = shap.TreeExplainer(
    model
)


# ============================================================
# 21. CALCULATE SHAP
# ============================================================

print(
    "\nCalculating SHAP values..."
)

shap_values = explainer.shap_values(
    X_shap
)


# ============================================================
# 22. SHAP FORMAT
# ============================================================

if isinstance(
    shap_values,
    list
):

    print(
        "SHAP returned list format."
    )

    shap_array = np.stack(
        shap_values,
        axis=2
    )

else:

    print(
        "SHAP returned ndarray format."
    )

    shap_array = np.asarray(
        shap_values
    )


print(
    "\nSHAP shape:",
    shap_array.shape
)


# ============================================================
# 23. HANDLE DIFFERENT SHAP VERSIONS
# ============================================================

n_samples = len(X_shap)

n_features = X_shap.shape[1]

n_classes = len(model.classes_)


# Expected:
#
# samples × features × classes
#
# OR sometimes:
#
# samples × classes × features


if shap_array.shape == (
    n_samples,
    n_features,
    n_classes
):

    print(
        "SHAP format = samples × features × classes"
    )

elif shap_array.shape == (
    n_samples,
    n_classes,
    n_features
):

    print(
        "SHAP format = samples × classes × features"
    )

    shap_array = np.transpose(
        shap_array,
        (0, 2, 1)
    )

else:

    raise ValueError(
        f"""
        Unexpected SHAP shape:

        {shap_array.shape}

        Expected one of:

        ({n_samples}, {n_features}, {n_classes})

        OR

        ({n_samples}, {n_classes}, {n_features})
        """
    )


print(
    "Final SHAP shape:",
    shap_array.shape
)


# ============================================================
# 24. GLOBAL SHAP IMPORTANCE
# ============================================================

global_importance = pd.DataFrame({

    "feature":
        X_shap.columns,

    "mean_abs_shap":
        np.abs(shap_array).mean(
            axis=(0, 2)
        ),

    "max_abs_shap":
        np.abs(shap_array).max(
            axis=(0, 2)
        )

})


global_importance = (
    global_importance
    .sort_values(
        "mean_abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


global_importance["rank"] = (
    np.arange(
        1,
        len(global_importance) + 1
    )
)


print("\n" + "=" * 80)
print("TOP GLOBAL SHAP FEATURES")
print("=" * 80)

print(
    global_importance.head(30).to_string(
        index=False
    )
)


# ============================================================
# 25. SAVE GLOBAL IMPORTANCE
# ============================================================

OUTPUT_DIR = "multiclass_shap_results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

global_importance.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "global_shap_importance.csv"
    ),

    index=False
)


# ============================================================
# 26. GLOBAL SHAP BAR
# ============================================================

global_abs_shap = np.abs(
    shap_array
).mean(
    axis=2
)


plt.figure(
    figsize=(12, 10)
)

shap.summary_plot(

    global_abs_shap,

    X_shap,

    plot_type="bar",

    max_display=30,

    show=False

)

plt.title(
    "Global SHAP Feature Importance"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "global_shap_bar.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()


# ============================================================
# 27. LIGHTGBM IMPORTANCE
# ============================================================

lgb_importance = pd.DataFrame({

    "feature":
        X_train.columns,

    "lightgbm_importance":
        model.feature_importances_

})


comparison = (

    global_importance

    .merge(
        lgb_importance,
        on="feature",
        how="left"
    )

)


comparison.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "lightgbm_vs_shap.csv"
    ),

    index=False
)


# ============================================================
# 28. CLASS-SPECIFIC SHAP
# ============================================================

class_records = []


for class_index, product in enumerate(
    model.classes_
):

    class_shap = shap_array[
        :,
        :,
        class_index
    ]

    temp = pd.DataFrame({

        "product_id":
            product,

        "feature":
            X_shap.columns,

        "mean_abs_shap":
            np.abs(
                class_shap
            ).mean(axis=0),

        "mean_shap":
            class_shap.mean(axis=0)

    })

    temp = temp.sort_values(
        "mean_abs_shap",
        ascending=False
    )

    class_records.append(
        temp
    )


class_importance = pd.concat(
    class_records,
    ignore_index=True
)


class_importance.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "product_class_shap_importance.csv"
    ),

    index=False
)


# ============================================================
# 29. TOP PRODUCTS
# ============================================================

top_products = (
    y.value_counts()
    .head(10)
    .index
)


print("\n" + "=" * 80)
print("TOP PRODUCTS")
print("=" * 80)

print(
    top_products.tolist()
)


# ============================================================
# 30. PRODUCT-SPECIFIC SHAP PLOTS
# ============================================================

for product in top_products:

    class_index = np.where(
        model.classes_ == product
    )[0][0]

    class_shap = shap_array[
        :,
        :,
        class_index
    ]

    plt.figure(
        figsize=(12, 10)
    )

    shap.summary_plot(

        class_shap,

        X_shap,

        max_display=20,

        show=False

    )

    plt.title(
        f"SHAP - Product {product}"
    )

    plt.tight_layout()

    filename = (
        f"shap_product_{str(product)}.png"
    )

    plt.savefig(

        os.path.join(
            OUTPUT_DIR,
            filename
        ),

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()


# ============================================================
# 31. LOCAL EXPLANATION
# ============================================================

local_position = 0

local_index = X_shap.index[
    local_position
]

X_single = X_shap.iloc[
    [local_position]
]

actual_product = y.loc[
    local_index
]

prediction = model.predict(
    X_single
)[0]

probabilities = model.predict_proba(
    X_single
)[0]

predicted_class_index = np.argmax(
    probabilities
)

predicted_probability = probabilities[
    predicted_class_index
]


print("\n" + "=" * 80)
print("LOCAL SHAP EXPLANATION")
print("=" * 80)

print(
    "Actual product:",
    actual_product
)

print(
    "Predicted product:",
    prediction
)

print(
    "Probability:",
    round(
        predicted_probability,
        4
    )
)


# ============================================================
# 32. LOCAL SHAP VALUES
# ============================================================

local_shap = shap_array[
    local_position,
    :,
    predicted_class_index
]


expected_value = explainer.expected_value


if isinstance(
    expected_value,
    (list, np.ndarray)
):

    base_value = expected_value[
        predicted_class_index
    ]

else:

    base_value = expected_value


# ============================================================
# 33. WATERFALL
# ============================================================

explanation = shap.Explanation(

    values=local_shap,

    base_values=base_value,

    data=X_single.iloc[0],

    feature_names=X_shap.columns

)


shap.plots.waterfall(

    explanation,

    max_display=20

)


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "local_waterfall.png"
    ),

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 34. LOCAL FEATURE TABLE
# ============================================================

local_table = pd.DataFrame({

    "feature":
        X_shap.columns,

    "feature_value":
        X_single.iloc[0].values,

    "shap_value":
        local_shap

})


local_table["direction"] = np.where(

    local_table["shap_value"] > 0,

    "Pushes toward predicted product",

    "Pushes away from predicted product"

)


local_table = local_table.sort_values(

    "shap_value",

    ascending=False

)


local_table.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "local_shap_explanation.csv"
    ),

    index=False
)


print("\nTop positive contributors:")

print(
    local_table.head(20).to_string(
        index=False
    )
)


print("\nTop negative contributors:")

print(
    local_table.tail(20).to_string(
        index=False
    )
)


# ============================================================
# 35. FINAL
# ============================================================

print("\n")
print("=" * 80)
print("COMPLETE MULTICLASS SHAP ANALYSIS FINISHED")
print("=" * 80)

print(
    "\nClasses:",
    NUM_CLASSES
)

print(
    "Features:",
    X.shape[1]
)

print(
    "Train rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

print(
    "SHAP rows:",
    len(X_shap)
)

print(
    "\nOutput directory:",
    OUTPUT_DIR
)

DATA VALIDATION
Original shape: (0, 1030)
Columns: ['purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity', 'last_p25_quantity', 'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity', 'last_p95_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_p25_spend', 'last_p50_spend', 'last_p75

ValueError: 
        DATAFRAME HAS ZERO ROWS AFTER PREPROCESSING.

        This means every product_id has fewer than
        2 observations.

        Check:

            sales[TARGET].value_counts()

        If you really have ~500 products but only one
        row per product, you CANNOT train a multiclass
        model to predict product_id from that dataset.
        

In [ ]:
#%pip install matplotlib shap lightgbm scikit-learn